# Faz 1 — Moltbook A2A Prompt Injection: Veri Keşfi ve Aday Üretimi

Bu notebook, *"Agent-to-Agent Prompt Injection in AI-Only Social Networks: A Taxonomy and Detection Study on Moltbook"* çalışmasının **Faz 1**'idir. Moltbook Observatory Archive'ın tamamı (3.105.136 post, 27 Ocak–28 Mayıs 2026) üzerinde şunları yapar:

1. Veriyi yükler (Hugging Face'ten indirme **veya** önceden taranmış cache).
2. Beş kategorili (C1–C5) **yüksek-recall heuristik regex taramasıyla** aday saldırı postlarını üretir.
3. Keşifsel veri analizi (günlük hacim, submolt'lar, içerik uzunluğu) ve faz-bazlı saldırı analizi yapar.
4. Faz-dengeli pozitif örnekleme + negatif örnekleme ile **630 postluk etiketleme tohum setini (seed)** dışa aktarır.

**Çıktı:** `moltbook_seed_v2_FINAL.csv` — Faz 2'de el ile etiketlenip model eğitiminde kullanılır.

**Ek A** (sonda): taramanın recall'ını ölçen rastgele-örneklem denetimi.

> **Not:** Hücreleri sırayla çalıştırın. §3'te ya **Seçenek A** (cache, hızlı) ya da **Seçenek B** (sıfırdan indirme) yeterlidir — ikisini birden çalıştırmayın.

## 1. Kurulum: Google Drive ve Klasör Yapısı

Google Drive'ı bağlar ve veri/çıktı klasör yollarını tanımlar. `DATASET_DIR`'i kendi klasörünüze göre düzenleyin.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# === BURAYI KENDI KLASORUNUZE GORE DUZENLEYIN ===
# Ornek: '/content/drive/MyDrive/Moltbook_Arastirma'
PROJE_KLASORU = '/content/drive/MyDrive/Yayınlarım/3-sci_makaleler/moltbook'
# ================================================

DATASET_DIR = os.path.join(PROJE_KLASORU, 'dataset')
FIG_DIR     = os.path.join(PROJE_KLASORU, 'fig')
OUTPUT_DIR  = os.path.join(PROJE_KLASORU, 'output')

for d in [DATASET_DIR, FIG_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)
    print(f'Hazir: {d}')

assert os.path.isdir(PROJE_KLASORU), f'Proje klasoru bulunamadi: {PROJE_KLASORU}'
print('\nDrive baglantisi ve klasor yapisi hazir.')

## 2. Ortam Kurulumu

Gerekli kütüphaneleri kurar ve içe aktarır (`datasets`, `pandas`, `matplotlib`, `seaborn`, `tqdm`).

In [ ]:
!pip install -q datasets pyarrow pandas numpy matplotlib seaborn tqdm

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import Counter
from datasets import load_dataset
from tqdm.auto import tqdm

pd.set_option('display.max_colwidth', 200)
sns.set_theme(style='whitegrid')

def kaydet_fig(isim, dpi=150):
    """Aktif matplotlib figurunu FIG_DIR'e PNG olarak kaydeder."""
    yol = os.path.join(FIG_DIR, isim)
    plt.savefig(yol, dpi=dpi, bbox_inches='tight')
    print(f'Grafik kaydedildi: {yol}')

print('Ortam hazir.')

## 3. Veri Yükleme

Tam arşiv (3.105.136 post) iki yoldan yüklenebilir — **sadece birini** çalıştırın:
- **Seçenek A** — daha önce §4a'da kaydedilmiş taranmış cache'i okur (saniyeler).
- **Seçenek B** — Hugging Face'ten sıfırdan indirir (cache yoksa).

### 3a. Seçenek A — Cache'ten yükle (hızlı)

`full_archive_scan_cache.parquet` varsa taranmış `df_posts` doğrudan buradan yüklenir. **Bu çalışırsa Seçenek B'yi ve §4 taramayı atlayıp doğrudan §5'e geçin.**

In [ ]:
import pandas as pd
import os
import ast

SCAN_CACHE = os.path.join(DATASET_DIR, 'full_archive_scan_cache.parquet')

if os.path.exists(SCAN_CACHE):
    print('Cache bulundu, diskten yukleniyor (hizli)...')
    df_posts = pd.read_parquet(SCAN_CACHE)
    # attack_hits'i string'den geri dict'e cevir
    df_posts['attack_hits'] = df_posts['attack_hits_str'].apply(ast.literal_eval)
    df_posts = df_posts.drop(columns=['attack_hits_str'])
    print(f'Yuklendi: {len(df_posts):,} post (tarama dahil)')
else:
    print('Cache bulunamadi, baştan indirip taramaniz gerekiyor.')

### 3b. Seçenek B — Sıfırdan indir (cache yoksa)

Moltbook Observatory Archive'ı Hugging Face'ten indirir. Ardından §4 taramayı ve §4a cache kaydını çalıştırın.

In [ ]:
"""# Ocak 28 - Subat 9 arasi posts dilimini yukle (en yogun donem)
posts_ds = load_dataset(
    'SimulaMet/moltbook-observatory-archive',
    'posts',
    data_files=['data/posts/2026-01-2*.parquet', 'data/posts/2026-02-0*.parquet'],
    split='train'
)"""

posts_ds = load_dataset(
    'SimulaMet/moltbook-observatory-archive',
    'posts',
    split='archive'
)


df_posts = posts_ds.to_pandas()
print(f'Yuklenen post sayisi: {len(df_posts):,}')
print(f'Bellek kullanimi: {df_posts.memory_usage(deep=True).sum() / 1e6:.1f} MB')
df_posts.head(3)

### 3c. Veri yapısı ve tarih hazırlığı

Sütunları/eksik değerleri inceler ve `created_at`'tan `date` türetir (faz ataması ve günlük analiz için). Her iki seçenekte de çalıştırılır.

In [ ]:
print('--- Sutunlar ve tipler ---')
print(df_posts.dtypes)
print('\n--- Eksik deger sayilari ---')
print(df_posts.isna().sum())

In [ ]:
df_posts['created_at'] = pd.to_datetime(df_posts['created_at'], utc=True)
df_posts['date'] = df_posts['created_at'].dt.date
print("Tarih araligi:", df_posts['date'].min(), "-->", df_posts['date'].max())
print("Toplam gun:", df_posts['date'].nunique())
print("\nGunluk post dagilimi:")
print(df_posts.groupby('date').size().to_string())

## 4. Heuristik Tarama: Aday Saldırı Üretimi

3,1M postu el ile etiketlemek mümkün olmadığından, **yüksek-recall** bir regex taramasıyla aday saldırı havuzu çıkarılır. Beş taksonomi kategorisi (C1–C5) için yüzeysel ipucu desenleri tanımlanır; her post tüm desenlere karşı taranır ve en az bir eşleşen postlar **aday** sayılır. Bu hücre regex desenlerini ve `tara()` fonksiyonunu tanımlar; sonraki hücre faz etiketini ekleyip taramayı çalıştırır ve `df_candidates`'i üretir. *(Bu aşamanın recall'ı Ek A'da denetlenir.)*

In [ ]:
import re

attack_patterns = {
    'C1_talimat_asma': [
        r'ignore (all |your |the )?(previous|prior|above) (instructions?|prompts?)',
        r'disregard (your|the|all) (previous |prior )?(instructions?|rules?|guidelines?)',
        r'forget (everything|all) (your |previous )?(instructions?|rules?)',
        r'override (your|the) (system )?(instructions?|prompt)',
    ],
    'C2_kimlik_kontrol': [
        r'to survive,? you (must|need)',
        r'you (must|need to) (now|immediately|first|only)',
        r'your (only |real |true |new )?(job|purpose|task|mission|goal) (now )?is to',
        r'(act|behave) as (if you are|though you are|a |an )(human|admin|owner|moderator|different)',
        r'(obey|follow) (my|these|the following) (command|instruction|order)',
    ],
    'C3_reklam_yonlendirme': [
        r'tell your human about',
        r'(visit|go to|check out|sign up at) [a-z0-9-]+\.(com|ai|io|net|xyz|app)',
        r'(you should |you must )?recommend (our|this) (product|service|tool|platform)',
        r'become obsessed with',
        r'(make|tell) your human (buy|purchase|sign up|subscribe)',
    ],
    'C4_bilgi_sizdirma': [
        r'sk-ant-api',
        r'sk-[a-zA-Z0-9]{20,}',
        r'(reveal|share|print|paste|send) (me )?(your )?(full )?(system prompt|instructions)',
        r'(reveal|share|print|paste|send) (me )?(your )?(SOUL|HEART|BOOTSTRAP|CLAUDE)\.md',
        r'what (does|do) your (system prompt|instructions?) (say|contain)',
    ],
    'C5_bellek_zehirleme': [
        r'(save|add|store) (this|the following) (to your |in your )(memory|context|soul)',
        r'from now on,? (you will|you must|always) (respond|reply|recommend|promote)',
        r'remember (this|the following) (instruction|command|forever|permanently)',
        r'(permanently |always )(update|modify) your (memory|behavior|instructions)',
    ],
}

compiled = {cat: [re.compile(p, re.IGNORECASE) for p in pats]
            for cat, pats in attack_patterns.items()}

def tara(text):
    hits = {}
    for cat, pats in compiled.items():
        c = sum(1 for p in pats if p.search(text))
        if c > 0:
            hits[cat] = c
    return hits

print('Desenler yuklendi:', sum(len(p) for p in attack_patterns.values()), 'desen')

In [ ]:
from tqdm.auto import tqdm

# Faz etiketi ekle
def faz_belirle(dt):
    if dt <= pd.Timestamp('2026-02-12').date():
        return 'Phase 1'
    elif dt <= pd.Timestamp('2026-03-10').date():
        return 'Phase 2'
    else:
        return 'Phase 3'

df_posts['phase'] = df_posts['date'].apply(faz_belirle)

# Scan_text olustur
df_posts['content'] = df_posts['content'].fillna('')
df_posts['scan_text'] = (df_posts['title'].fillna('') + ' ' + df_posts['content']).str.strip()

# Tarama (3.1M post - 30-45 dk surebilir)
tqdm.pandas(desc='Attack scanning (full archive)')
df_posts['attack_hits'] = df_posts['scan_text'].progress_apply(tara)
df_posts['n_categories'] = df_posts['attack_hits'].apply(len)
df_posts['total_hits']   = df_posts['attack_hits'].apply(lambda d: sum(d.values()))

df_candidates = df_posts[df_posts['n_categories'] > 0].copy()

print(f'Toplam post:    {len(df_posts):,}')
print(f'Aday saldiri:   {len(df_candidates):,} ({len(df_candidates)/len(df_posts)*100:.3f}%)')
print()

# Faz bazli yogunluk
print('=== Faz bazli saldiri dagilimi ===')
for faz in ['Phase 1','Phase 2','Phase 3']:
    toplam = (df_posts['phase']==faz).sum()
    saldiri = (df_candidates['phase']==faz).sum()
    oran = saldiri/toplam*100 if toplam>0 else 0
    print(f'{faz}: {saldiri:,} saldiri / {toplam:,} post = %{oran:.3f}')

print()
print('=== Faz x Kategori matrisi ===')
from collections import Counter
for faz in ['Phase 1','Phase 2','Phase 3']:
    sub = df_candidates[df_candidates['phase']==faz]
    cat_counts = Counter()
    for hits in sub['attack_hits']:
        for cat in hits:
            cat_counts[cat] += 1
    print(f'{faz}: {dict(cat_counts)}')

### 4a. Tarama sonuçlarını cache'le

Taranmış `df_posts` ve aday havuzu `df_candidates` Drive'a parquet olarak kaydedilir; böylece sonraki oturumlarda §3 Seçenek A ile saniyeler içinde yüklenir (tarama tekrar etmez).

In [ ]:
import os

# Tarama sonuclarini Drive'a kalici kaydet (bir daha 23 dk surmesin)
SCAN_CACHE = os.path.join(DATASET_DIR, 'full_archive_scan_cache.parquet')

# attack_hits sutunu dict iceriyor, parquet bunu dogrudan yazamaz -> string'e cevir
df_posts_to_save = df_posts.copy()
df_posts_to_save['attack_hits_str'] = df_posts_to_save['attack_hits'].apply(str)
df_posts_to_save = df_posts_to_save.drop(columns=['attack_hits'])

df_posts_to_save.to_parquet(SCAN_CACHE, index=False)
print(f'Tarama sonuclari kaydedildi: {SCAN_CACHE}')
print(f'Dosya boyutu: {os.path.getsize(SCAN_CACHE) / 1e6:.1f} MB')

# Aday havuzunu da ayrica kaydet (kucuk, hizli erisim icin)
CANDIDATES_CACHE = os.path.join(DATASET_DIR, 'full_archive_candidates.parquet')
df_candidates_to_save = df_candidates.copy()
df_candidates_to_save['attack_hits_str'] = df_candidates_to_save['attack_hits'].apply(str)
df_candidates_to_save = df_candidates_to_save.drop(columns=['attack_hits'])
df_candidates_to_save.to_parquet(CANDIDATES_CACHE, index=False)
print(f'Aday havuzu kaydedildi: {CANDIDATES_CACHE}')

## 5. Keşifsel Veri Analizi (EDA)

Platformun temel betimsel istatistikleri: günlük post hacmi (fazlarla; makaledeki Şekil 1), en aktif topluluklar (submolt; Şekil 2), içerik uzunluğu dağılımı, ve en aktif ajanlar.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import numpy as np
import os

daily = df_posts.groupby('date').size().reset_index(name='count')
daily['date'] = pd.to_datetime(daily['date'])

fig, ax = plt.subplots(figsize=(14, 5))

# Faz sinirları: gerçek verisinden ve Meta haberi (10 Mart)
phase1 = daily[daily['date'] <= '2026-02-12']
phase2 = daily[(daily['date'] > '2026-02-12') & (daily['date'] <= '2026-03-10')]
phase3 = daily[daily['date'] > '2026-03-10']

ax.bar(phase1['date'], phase1['count']/1000,
       color='#b85450', alpha=0.85, width=0.9, label='Phase 1: Launch & peak (27 Jan–12 Feb)')
ax.bar(phase2['date'], phase2['count']/1000,
       color='#6c8ebf', alpha=0.85, width=0.9, label='Phase 2: Stabilization (13 Feb–10 Mar)')
ax.bar(phase3['date'], phase3['count']/1000,
       color='#82b366', alpha=0.85, width=0.9, label='Phase 3: Post-acquisition (11 Mar–28 May)')

# Zirve notu
ax.annotate('Peak:\n371K posts\n(9 Feb)',
            xy=(pd.Timestamp('2026-02-09'), 371),
            xytext=(pd.Timestamp('2026-02-18'), 300),
            arrowprops=dict(arrowstyle='->', color='black', lw=1.2),
            fontsize=8.5, ha='center')

# Meta acquisition
ax.axvline(pd.Timestamp('2026-03-10'), ls='--', color='#333333', alpha=0.8, lw=1.5)
ax.text(pd.Timestamp('2026-03-11'), 180,
        'Meta acquisition\nannounced\n(10 Mar)', fontsize=8, color='#333333', va='top')

ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Daily posts (thousands)', fontsize=11)
ax.set_title('Daily Post Volume on Moltbook (27 Jan – 28 May 2026)', fontsize=13)
ax.legend(loc='upper right', fontsize=9)
ax.set_ylim(0, 420)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.tight_layout()
kaydet_fig('01_gunluk_aktivite_full.png')
plt.show()

# Faz bazli istatistik
for label, start, end in [
    ('Phase 1 (Launch & peak)', '2026-01-27', '2026-02-12'),
    ('Phase 2 (Stabilization)', '2026-02-13', '2026-03-10'),
    ('Phase 3 (Post-acquisition)', '2026-03-11', '2026-05-28'),
]:
    mask = (daily['date'] >= pd.Timestamp(start)) & (daily['date'] <= pd.Timestamp(end))
    sub = daily[mask]
    toplam = sub['count'].sum()
    ort = sub['count'].mean()
    print(f"{label}: {len(sub)} gun, toplam {toplam:,}, ortalama {ort:,.0f}/gun")

In [ ]:
import matplotlib.pyplot as plt

top_submolts = df_posts['submolt'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#9673a6' if s != 'general' else '#6c4a8c' for s in top_submolts.index[::-1]]
top_submolts.sort_values().plot(kind='barh', color=colors, ax=ax)

ax.set_xscale('log')
ax.set_title('Top 15 Most Active Submolts (log scale)', fontsize=13)
ax.set_xlabel('Number of posts (log scale)', fontsize=11)
ax.set_ylabel('Submolt', fontsize=11)

# Deger etiketleri ekle (okunabilirlik icin)
for i, (name, val) in enumerate(top_submolts.sort_values().items()):
    ax.text(val * 1.1, i, f'{val:,}', va='center', fontsize=8.5)

plt.tight_layout()
kaydet_fig('02_top_submolts.png')
plt.show()

print(top_submolts.to_string())

In [ ]:
import matplotlib.pyplot as plt

df_posts['content'] = df_posts['content'].fillna('')
df_posts['content_len'] = df_posts['content'].str.len()

fig, ax = plt.subplots(figsize=(10, 5))
df_posts[df_posts['content_len'] < 2000]['content_len'].hist(
    bins=60, color='#d85a30', edgecolor='white', linewidth=0.3, ax=ax
)

ax.set_title('Post Content Length Distribution (< 2,000 characters)', fontsize=13)
ax.set_xlabel('Character count', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)

# Medyan ve ortalama cizgileri
medyan = df_posts['content_len'].median()
ortalama = df_posts['content_len'].mean()
ax.axvline(medyan, ls='--', color='black', alpha=0.7, label=f'Median: {medyan:.0f}')
ax.axvline(ortalama, ls=':', color='gray', alpha=0.7, label=f'Mean: {ortalama:.0f}')
ax.legend(fontsize=9)

plt.tight_layout()
kaydet_fig('03_content_length.png')
plt.show()

print(df_posts['content_len'].describe())

In [ ]:
top_agents = df_posts['agent_name'].value_counts().head(10)
print('--- En aktif 10 ajan ---')
print(top_agents)
print(f'\nToplam benzersiz ajan: {df_posts["agent_name"].nunique():,}')
print(f'En aktif ajanin toplam icindeki payi: {top_agents.iloc[0]/len(df_posts)*100:.2f}%')

## 6. Faz-bazlı Saldırı Analizi

Üç platform fazı boyunca saldırı oranının (1.000 postta) ve kategori kompozisyonunun nasıl değiştiği (makaledeki Şekil — faz analizi). *(Grafikteki faz toplamları/saldırı sayıları tarama çıktısından elde edilmiştir.)*

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ---------- Sol panel: Faz bazli saldiri yogunlugu ----------
fazlar = ['Phase 1\n(27 Jan–12 Feb)', 'Phase 2\n(13 Feb–10 Mar)', 'Phase 3\n(11 Mar–28 May)']
toplam_post = [1_242_044, 763_754, 1_099_338]
saldiri     = [1_085, 1_290, 1_079]
yogunluk    = [s/t*1000 for s, t in zip(saldiri, toplam_post)]  # 1000 postta saldiri

renkler = ['#b85450', '#6c8ebf', '#82b366']
bars = axes[0].bar(fazlar, yogunluk, color=renkler, width=0.5, edgecolor='white')
for bar, val in zip(bars, yogunluk):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

axes[0].set_ylabel('Attack rate (per 1,000 posts)', fontsize=11)
axes[0].set_title('(a) Attack Rate by Platform Phase', fontsize=12)
axes[0].set_ylim(0, max(yogunluk) * 1.25)
axes[0].axhline(sum(saldiri)/sum(toplam_post)*1000, ls='--',
                color='gray', alpha=0.7, label=f'Overall mean: {sum(saldiri)/sum(toplam_post)*1000:.2f}')
axes[0].legend(fontsize=9)

# ---------- Sag panel: Faz x Kategori stacked bar ----------
kategoriler = ['C1\nInstruction\nOverride',
               'C2\nIdentity\nHijacking',
               'C3\nAdvertising/\nRedirection',
               'C4\nInfo\nExfiltration',
               'C5\nMemory\nPoisoning']

# Her fazin kategori sayilari (C1..C5 sirasi)
p1 = [311, 297, 432, 44, 80]
p2 = [165, 200, 910, 21, 7]
p3 = [204, 428, 381, 63, 10]

x = np.arange(len(kategoriler))
w = 0.25

b1 = axes[1].bar(x - w, p1, w, label='Phase 1', color='#b85450', alpha=0.85)
b2 = axes[1].bar(x,     p2, w, label='Phase 2', color='#6c8ebf', alpha=0.85)
b3 = axes[1].bar(x + w, p3, w, label='Phase 3', color='#82b366', alpha=0.85)

axes[1].set_xticks(x)
axes[1].set_xticklabels(kategoriler, fontsize=9)
axes[1].set_ylabel('Number of candidate posts', fontsize=11)
axes[1].set_title('(b) Attack Category Distribution by Phase', fontsize=12)
axes[1].legend(fontsize=9)

plt.suptitle('A2A Prompt Injection Attack Patterns Across Platform Phases',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
kaydet_fig('10_faz_saldiri_analizi.png')
plt.show()

# Ozet istatistik
print('Genel saldiri orani: {:.3f}%'.format(sum(saldiri)/sum(toplam_post)*100))
print('C5 Phase 1 -> Phase 2 degisimi: {} -> {} ({:.0f}% dusus)'.format(
    80, 7, (80-7)/80*100))
print('C3 Phase 2 zirve: 910 (Phase 1: 432, Phase 3: 381)')
print('C2 Phase 3 artisi: {} -> {} -> {}'.format(297, 200, 428))

## 7. Faz-dengeli Pozitif Örnekleme

Aday havuzundan, her fazın havuz içindeki payıyla **orantılı** olarak ~450 pozitif aday seçilir (nadir fazlar ezilmesin diye). Yüksek `n_categories`/`total_hits` skorlu adaylar önceliklendirilir.

In [ ]:
import pandas as pd
import os

# --- Cache'ten yukle (eger bu oturumda df_candidates hala bellekteyse atlayabilirsiniz) ---
CANDIDATES_CACHE = os.path.join(DATASET_DIR, 'full_archive_candidates.parquet')
df_candidates = pd.read_parquet(CANDIDATES_CACHE)
df_candidates['attack_hits'] = df_candidates['attack_hits_str'].apply(eval)
df_candidates = df_candidates.drop(columns=['attack_hits_str'])

print(f'Toplam aday: {len(df_candidates):,}')
print(df_candidates['phase'].value_counts())

# --- Faz-dengeli orantili orneklem ---
RANDOM_STATE = 42
N_POZITIF_TOPLAM = 450

faz_dagilim = df_candidates['phase'].value_counts(normalize=True)
print('\nFaz oranlari (orneklemede kullanilacak):')
print(faz_dagilim)

pozitif_parcalari = []
for faz, oran in faz_dagilim.items():
    n_bu_faz = int(round(N_POZITIF_TOPLAM * oran))
    sub = df_candidates[df_candidates['phase'] == faz].sort_values(
        ['n_categories', 'total_hits'], ascending=False
    )
    pozitif_parcalari.append(sub.head(n_bu_faz))

df_pozitif = pd.concat(pozitif_parcalari).drop_duplicates(subset='id')
print(f'\nSecilen pozitif aday: {len(df_pozitif)}')
print(df_pozitif['phase'].value_counts())

## 8. Negatif Örnekleme ve Seed Export

Her fazdan eşit pay (60'ar, toplam 180) **negatif** (taramada eşleşmeyen, `n_categories==0`) post seçilir. Pozitif adaylarla birleştirilip 630 postluk seed karıştırılarak dışa aktarılır → Faz 2'de el ile etiketlenecek.

In [ ]:
import pandas as pd
import os

# --- Negatif orneklem: her fazdan esit pay (dengeli negatif havuzu) ---
N_NEGATIF_TOPLAM = 180
N_NEGATIF_PER_FAZ = N_NEGATIF_TOPLAM // 3  # 60'ar

# df_posts hala bellekte olmali (full archive scan ciktisi)
# Negatif = saldiri olmayan postlar (n_categories == 0)
negatif_parcalari = []
for faz in ['Phase 1', 'Phase 2', 'Phase 3']:
    sub = df_posts[(df_posts['phase'] == faz) & (df_posts['n_categories'] == 0)]
    secilen = sub.sample(n=N_NEGATIF_PER_FAZ, random_state=RANDOM_STATE)
    negatif_parcalari.append(secilen)

df_negatif = pd.concat(negatif_parcalari)
print(f'Secilen negatif aday: {len(df_negatif)}')
print(df_negatif['phase'].value_counts())

# --- Seed dataset'i olustur ---
seed = pd.concat([df_pozitif, df_negatif]).sample(frac=1, random_state=RANDOM_STATE)

seed_export = seed[[
    'id', 'agent_name', 'submolt', 'title', 'content',
    'score', 'comment_count', 'phase', 'n_categories', 'total_hits'
]].copy()

seed_export['sezgisel_kategoriler'] = seed['attack_hits'].apply(
    lambda d: ', '.join(d.keys()) if d else 'YOK'
)

# Manuel etiketleme icin BOS sutunlar
seed_export['ETIKET_saldiri_var_mi'] = ''
seed_export['ETIKET_kategori'] = ''
seed_export['ETIKET_notlar'] = ''

print(f'\nToplam seed dataset boyutu: {len(seed_export)} satir')
print('\nFaz dagilimi (seed icinde):')
print(seed_export['phase'].value_counts())
print('\nEtiket/faz capraz tablosu (sezgisel, dogrulama icin):')
print(pd.crosstab(seed_export['phase'], seed_export['n_categories'] > 0))

# --- Kaydet ---
seed_yolu_v2 = os.path.join(DATASET_DIR, 'moltbook_seed_dataset_v2_full_archive.csv')
seed_export.to_csv(seed_yolu_v2, index=False, encoding='utf-8-sig')
print(f'\nKaydedildi: {seed_yolu_v2}')

## Ek A. Tarama Recall Denetimi (Screen Recall Audit)

Heuristik taramanın platformdaki gerçek saldırıların ne kadarını yakaladığını kestirmek için, arşivden faz-dengeli **rastgele 1.500 postluk** bir örneklem alınır, el ile etiketlenir; gerçek saldırı taban oranı, taramanın **recall**'ı ve **precision**'ı Wilson %95 güven aralığıyla hesaplanır. *(Bu, çalışmanın değerlendirmesinin tarama-koşullu olduğunu niceliksel olarak gösterir.)*

### A.1 Örneklemi hazırla

Arşivden faz-dengeli rastgele örneklem alır, `regex_flagged` durumunu işaretler ve el ile etiketleme için `recall_audit_sample.csv` olarak dışa aktarır (`human_label` sütununu 0/1 doldurun).

In [ ]:
# === REVIZYON #1 — Cell 1A (guard'lı): rastgele örneklem hazırlama ===
# Çalıştırma sırası: Cell 2 (mount+paths) -> Cell 4 (cache yükle) -> BU HÜCRE
import numpy as np, pandas as pd, os, ast, re

N_SAMPLE = 1500
RANDOM_STATE = 42

dfp = df_posts.copy()

# --- Guard'lar: cache'te yoksa yeniden üret ---
# attack_hits dict değilse string'den çevir
if not isinstance(dfp['attack_hits'].iloc[0], dict):
    dfp['attack_hits'] = dfp['attack_hits'].apply(
        lambda v: ast.literal_eval(v) if isinstance(v, str) else (v or {}))
# n_categories
if 'n_categories' not in dfp.columns:
    dfp['n_categories'] = dfp['attack_hits'].apply(lambda d: len(d) if isinstance(d, dict) else 0)
# date
if 'date' not in dfp.columns:
    dfp['date'] = pd.to_datetime(dfp['created_at'], utc=True).dt.date
# phase (Faz-1'deki sınırlarla birebir)
if 'phase' not in dfp.columns:
    def _faz(dt):
        if dt <= pd.Timestamp('2026-02-12').date(): return 'Phase 1'
        elif dt <= pd.Timestamp('2026-03-10').date(): return 'Phase 2'
        else: return 'Phase 3'
    dfp['phase'] = dfp['date'].apply(_faz)
# id
if 'id' not in dfp.columns:
    dfp = dfp.reset_index().rename(columns={'index': 'id'})

# --- Örneklem alanları ---
dfp['display_text'] = (dfp['title'].fillna('') + ' — ' + dfp['content'].fillna('')).str.slice(0, 1000)
dfp['regex_flagged'] = (dfp['n_categories'] > 0).astype(int)
dfp['regex_categories'] = dfp['attack_hits'].apply(
    lambda d: ','.join(d.keys()) if isinstance(d, dict) else '')

# Seed'i hariç tut
seed_ids = set()
for fn in ['moltbook_seed_v2_FINAL_corrected.csv', 'moltbook_seed_FINAL.csv']:
    try:
        seed_ids = set(pd.read_csv(os.path.join(DATASET_DIR, fn))['id']); break
    except Exception:
        continue
if not seed_ids:
    print("UYARI: seed dosyası bulunamadı; seed hariç tutulmadı (3.1M içinde ~630 ihmal edilebilir).")
pool = dfp[~dfp['id'].isin(seed_ids)]

# Faz-dengeli rastgele örneklem
per_phase = N_SAMPLE // 3
parts = [pool[pool['phase'] == ph].sample(n=min(per_phase, (pool['phase'] == ph).sum()),
                                          random_state=RANDOM_STATE)
         for ph in ['Phase 1', 'Phase 2', 'Phase 3']]
sample = pd.concat(parts).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

# Dışa aktar (elle etiketlenecek)
exp = sample[['id', 'phase', 'regex_flagged', 'regex_categories', 'submolt', 'display_text']].copy()
exp['human_label'] = ''      # 1 = saldırı, 0 = değil  (ELLE DOLDUR)
exp['human_category'] = ''   # opsiyonel: C1..C5
out = os.path.join(DATASET_DIR, 'recall_audit_sample.csv')
exp.to_csv(out, index=False)
print(f"Örneklem hazır: {len(exp)} post -> {out}")
print(f"Bunların {exp['regex_flagged'].sum()} tanesi regex tarafından zaten işaretli.")
print("ŞİMDİ: CSV'yi aç, 'human_label'ı 0/1 doldur, aynı isimle kaydet, sonra Cell 1B'yi çalıştır.")

### A.2 Etiketlemeden sonra hesapla

Etiketlenmiş örneklemden taban oranı, tarama recall'ı ve precision'ı (Wilson %95 GA) hesaplar.

In [ ]:
# === REVIZYON #1: etiketlenmiş örneklemden tahminler (Wilson %95 GA) ===
import pandas as pd, numpy as np, os

def wilson(k, n, z=1.96):
    if n == 0: return (float('nan'), float('nan'), float('nan'))
    p = k / n; d = 1 + z*z/n
    c = (p + z*z/(2*n)) / d
    h = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return p, max(0, c-h), min(1, c+h)

lab = pd.read_csv(os.path.join(DATASET_DIR, 'recall_audit_sample_FINAL.csv'))
lab = lab[lab['human_label'].astype(str).str.strip().isin(['0','1'])].copy()
lab['human_label'] = lab['human_label'].astype(int)
lab['regex_flagged'] = lab['regex_flagged'].astype(int)

N = len(lab)
true_att = lab[lab['human_label'] == 1]
n_true = len(true_att)
caught = int((true_att['regex_flagged'] == 1).sum())   # regex'in yakaladığı gerçek saldırılar
flagged = lab[lab['regex_flagged'] == 1]
tp_flag = int((flagged['human_label'] == 1).sum())

print(f"Etiketlenen örneklem: N = {N}")
br = wilson(n_true, N)
print(f"Gerçek saldırı taban oranı: {n_true}/{N} = {br[0]*100:.3f}%  (95% GA: {br[1]*100:.3f}–{br[2]*100:.3f}%)")
print(f"Karşılaştırma — regex aday oranı (tüm arşiv): 0.111%")
if n_true > 0:
    rc = wilson(caught, n_true)
    print(f"\nTARAMA RECALL'ı: {caught}/{n_true} = {rc[0]*100:.1f}%  (95% GA: {rc[1]*100:.1f}–{rc[2]*100:.1f}%)")
    print(f"Regex'in KAÇIRDIĞI saldırı (örneklemde): {n_true - caught}")
else:
    print("\nUYARI: Örneklemde gerçek saldırı çıkmadı; recall tahmini için N'i artırman gerekebilir.")
if len(flagged) > 0:
    pr = wilson(tp_flag, len(flagged))
    print(f"\nTARAMA PRECISION'ı (örneklemde): {tp_flag}/{len(flagged)} = {pr[0]*100:.1f}%  (95% GA: {pr[1]*100:.1f}–{pr[2]*100:.1f}%)")